# Molecular docking on a Coherent Ising Machine

Two protein-ligand complexes -- PDB **1N2J** and **1LRH** -- are redocked on a
real **Coherent Ising Machine** (CIM) through the **Kaiwu SDK**. Each pose search is
written as a **QUBO** with the **Grid Point Matching** encoding and solved on the
photonic hardware in **quota mode at 8-bit precision**.

## Setup

In [ ]:
import os, sys
ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, ROOT)
for _m in [m for m in list(sys.modules) if m.startswith("qdock_kaiwu")]:
    del sys.modules[_m]                                      # pick up the latest package code

import sys, numpy as np
from types import SimpleNamespace
from qdock_kaiwu import backends, evaluate, io
from qdock_kaiwu.gpm import _matches_to_poses, GPMDock
from qdock_kaiwu.qubo import build_gpm_qubo

MOLS = ["1N2J", "1LRH"]
B = {p: dict(np.load(f"{ROOT}/results/{p}_cim.npz", allow_pickle=True)) for p in MOLS}
print("loaded bundles:", ", ".join(f"{p} ({int(B[p]['n_runs'])} pooled CIM runs)"
      for p in MOLS))

## 1. From structures to a QUBO

Docking starts from two structures: the **receptor** (the protein) and the **ligand** (the
small molecule to place inside it). We walk through one target, PDB **1N2J** -- load the
prepared receptor and read the ligand into AutoDock atom types with OpenBabel.

In [ ]:
import os, tempfile

p    = "1N2J"                                                # 1N2J or 1LRH
dock = GPMDock(backend="cim", workdir=os.path.join(tempfile.gettempdir(), f"qdock_{p}"))
dock.use_receptor_pdbqt(f"{ROOT}/data/{p}_receptor.pdbqt")  # prepared protein
dock.make_ligand([f"{ROOT}/data/{p}_ligand.mol2"])          # OpenBabel: mol2 -> AutoDock types
lig = dock.ligands[0]
print(f"receptor : {len(io.read_pdbqt(dock.receptor_pdbqt)['coords'])} atoms")
print(f"ligand   : {lig.n} atoms  {list(lig.ad_types)}")

### A box over the pocket -- you choose it

We do not search all of space; we lay a **regular grid of candidate positions** inside a
**docking box you define** -- a centre, a size in ångström and a grid spacing. `make_box_input`
builds that grid and runs **AutoGrid**, which scores every grid point: for each ligand atom
*type* it computes the van-der-Waals interaction energy with the whole receptor (negative =
favourable). The default box is centred on the ligand; change the centre or size to dock a
different region (keep the grid small enough that the variable count stays under ~1000).

In [ ]:
cx, cy, cz = lig.coords.mean(0)                             # box centre (default: ligand centroid)
box_size   = {"1N2J": (8, 8, 12), "1LRH": (10, 10, 10)}[p]  # Angstrom -- edit for a custom box
dock.make_box_input(cx, cy, cz, *box_size, grid_length=2.0) # lay the grid + run AutoGrid

print(f"box: {len(dock.box_coords)} grid points, 2 A spacing, size {box_size} A, "
      f"centred at ({cx:.1f}, {cy:.1f}, {cz:.1f})")

### Assembling the QUBO

Each favourable *(ligand atom, grid point)* pair becomes one binary variable,
$x_{a,s}=1 \iff$ atom $a$ sits at grid point $s$. The QUBO rewards good placements and
penalises inconsistent ones:

```
H(x) = Σ w[a,s]·x[a,s]                                  # AutoGrid reward   (diagonal)
     + K_dist · Σ ( ‖d_lig − d_grid‖ > c ) · x_p·x_q    # shape distortion  (off-diagonal)
     + K_mono · Σ ( same atom )            · x_p·x_q    # one atom, one point (off-diagonal)
```

- **w[a,s]** is the AutoGrid reward we just computed -- the only chemistry input;
- **K_dist** penalises a pair of matches that would stretch or squeeze the rigid ligand;
- **K_mono** forbids one atom from taking two grid points.

The next cell builds `Q` straight from those pieces -- the rewards on the diagonal, the two
penalties from the ligand/grid geometry in the upper triangle -- the same matrix the engine's
`build_gpm_qubo` produces. With the default box it is identical to the one behind the pooled
CIM results shown later.

In [ ]:
# every favourable (ligand atom, grid point) pair -> one binary variable; reward = its AutoGrid energy
w, atom, site = [], [], []
for i, t in enumerate(lig.ad_types):
    for pos, e in zip(*dock.grid_dict[t]):
        w.append(e); atom.append(i); site.append(int(pos))
w = np.array(w, float); atom, site = np.array(atom), np.array(site)
variables = np.column_stack([atom, site]); V = len(w)

# ligand internal distances vs grid internal distances, indexed per variable pair
pdist  = lambda X: np.linalg.norm(X[:, None] - X[None, :], axis=-1)
d_lig  = pdist(lig.coords)[np.ix_(atom, atom)]
d_grid = pdist(dock.box_coords)[np.ix_(site, site)]

# assemble Q: AutoGrid reward on the diagonal, the two penalties in the upper triangle
c, K_dist, K_mono = 2.565, 2.337, 39.530
Q    = np.diag(w)
pen  = (np.abs(d_lig - d_grid) > c) * K_dist                # placement distorts the rigid ligand
pen += (atom[:, None] == atom[None, :]) * K_mono            # one atom on two grid points
iu   = np.triu_indices(V, 1); Q[iu] += pen[iu]

Qe, _ = build_gpm_qubo(lig.coords, lig.ad_types, dock.grid_dict, dock.box_coords, c, K_dist, K_mono)
print(f"QUBO Q: {V}x{V} spins; reward w in [{w.min():.2f}, {w.max():.2f}] kcal/mol")
print("matches qdock_kaiwu.qubo.build_gpm_qubo:", np.allclose(Q, Qe))

## 2. From the QUBO to the CIM

The Coherent Ising Machine minimises an **Ising** Hamiltonian over ±1 spins, so the QUBO is
first converted with `kw.conversion.qubo_matrix_to_ising_matrix(Q)` -- this is the matrix the
hardware actually sees. The `V` variables become `V`+1 spins (one ancilla fixes the 0/1 ↔ ±1
gauge), and the converter folds in the sign so the machine's maximiser minimises our QUBO.
The conversion is exact: the Ising energy of any assignment equals its QUBO energy.

In [ ]:
import kaiwu as kw

ising, off = kw.conversion.qubo_matrix_to_ising_matrix(Q)   # the matrix the hardware sees
print(f"Ising matrix: {np.asarray(ising).shape[0]} spins  ( {V} variables + 1 gauge ancilla )")

# the conversion is exact -- QUBO and Ising energies agree for any assignment:
b = np.random.default_rng(0).integers(0, 2, V); s = np.concatenate([2*b - 1, [1]])[None, :]
H = float(kw.common.hamiltonian(ising, s).flatten()[0]) + off
print("QUBO energy preserved by the Ising form:", np.isclose(b @ Q @ b, H))

### The solve

`kw.cim.CIMOptimizer(task_mode="quota")` submits the Ising matrix to the photonic machine,
wrapped in `kw.cim.PrecisionReducer(precision=8, truncated_precision=8)` -- **8-bit**
precision with **no spin expansion**, so the spin count stays equal to the number of
variables (232-304 here, well under the ~1000-spin budget). `reducer.solve` returns the
spins, which are gauged back to binary matches. The next cell is the live submission.

In [ ]:
backends.init_license()                                                  # license from the environment

def cim_solve(ising, n_spins, task_name):
    cim = kw.cim.CIMOptimizer(task_name=task_name, wait=True, interval=2,
                              task_mode="quota", sample_number=300)       # quota mode
    reducer = kw.cim.PrecisionReducer(cim, precision=8, truncated_precision=8,
                                      only_feasible_solution=False)       # 8-bit, no spin expansion
    spins = np.asarray(reducer.solve(ising))                             # submit -> poll -> spins
    return [backends._spins_to_binary(s, n_spins) for s in spins]       # gauge +/-1 -> binary

reads = cim_solve(ising, V, "qdock_1N2J_GPM_quota_p8t8")
print(f"got {len(reads)} reads over {V} spins")

## 3. From spins to a pose

Every spin row is gauged to a binary match vector, the unique matches are ranked by
QUBO energy, and the matched atoms are Kabsch-superposed onto their grid points. The
**mRMSD** -- the smallest heavy-atom RMSD to the crystal ligand over the returned
poses -- measures the *sampling power*: does a near-native pose exist among them?

In [ ]:
def decode(d, reads=None):
    # Rank match sets by QUBO energy, then Kabsch-superpose each onto its grid points.
    Q = d["Q"].astype(float)
    ranked = (backends._rank_unique(reads, Q) if reads is not None
              else list(zip(d["energies"].tolist(), list(d["solutions"]))))
    lig = SimpleNamespace(coords=d["lig_coords"])
    poses, _ = _matches_to_poses(lig, d["variables"], d["box_coords"], ranked)
    poses = np.array(poses)
    rmsd = evaluate.pose_rmsds(poses, d["lig_coords"], d["lig_elements"])
    return poses, rmsd

live = {"Q": Q, "variables": variables, "box_coords": dock.box_coords,
        "lig_coords": lig.coords, "lig_elements": lig.elements}
poses, rmsd = decode(live, reads)
print(f"{p}: {len(poses)} poses from the CIM,  mRMSD = {rmsd.min():.2f} Å")

## 4. The same recipe on both targets

One fixed recipe -- quota mode, 8-bit, pooled runs -- applied to each complex.

In [ ]:
print(f"{'PDB':<6}{'spins':>7}{'grid/Å':>8}{'poses':>7}{'mRMSD/Å':>9}")
for p in MOLS:
    d = B[p]; _, r = decode(d)
    print(f"{p:<6}{int(d['n_vars']):>7}{float(d['grid_length']):>8.1f}"
          f"{len(r):>7}{r.min():>9.2f}")

## 5. Reproducibility across independent runs

The CIM is stochastic: each submission is an independent draw, and the near-native
pose surfaces in only a fraction of them, so the recipe pools several runs and keeps
the best. The table reports, per target, how many independent hardware runs reached a
docking-quality pose (< 2 Å) and a near-native pose (< 1 Å).

In [ ]:
print(f"{'PDB':<6}{'runs':>6}{'<2Å':>7}{'<1Å':>7}{'best':>7}{'median':>8}")
for p in MOLS:
    r = B[p]["per_run_rmsd"]
    print(f"{p:<6}{len(r):>6}{f'{int((r<2).sum())}/{len(r)}':>7}"
          f"{f'{int((r<1).sum())}/{len(r)}':>7}{r.min():>7.2f}{np.median(r):>8.2f}")

## 6. Scoring the poses with AutoDock Vina

A low RMSD means a good pose *exists* in the pool (sampling power). Whether a
physics-based scoring function can *select* it is a separate question (scoring power).
Each pooled pose is rescored with `AutoDock Vina --score_only` and the top-1 is the
lowest-affinity pose -- for both targets it is the near-native pose, so sampling and
scoring agree here. (Scores are bundled, so this cell needs no Vina binary.)

In [ ]:
print(f"{'PDB':<6}{'Vina top-1':>11}{'top-1 RMSD':>12}{'mRMSD':>8}{'crystal Vina':>14}")
for p in MOLS:
    d = B[p]; _, rmsd = decode(d)
    vina = d["vina_scores"]; i = int(np.nanargmin(vina))
    print(f"{p:<6}{vina[i]:>11.2f}{rmsd[i]:>12.2f}{rmsd.min():>8.2f}"
          f"{float(d['crystal_vina']):>14.2f}")
print("\n(Vina affinity in kcal/mol, lower = better; RMSD in Å to the crystal ligand)")

## 7. Best pose vs crystal

The lowest-RMSD CIM pose (coloured sticks) overlaid on the crystal ligand (blue).

In [ ]:
import matplotlib.pyplot as plt

EL = {'C': '#444', 'N': '#2c5fa8', 'O': '#cc3333', 'S': '#d4a000', 'P': '#d4a000',
      'ZN': '#9a6', 'CL': '#3a3'}

def bonds(xyz, el):
    keep = np.array([e != 'H' for e in el]); idx = np.where(keep)[0]; P = xyz[keep]
    return [(idx[i], idx[j]) for i in range(len(P)) for j in range(i + 1, len(P))
            if np.linalg.norm(P[i] - P[j]) < 1.95]

def draw(ax, crystal, docked, el, title):
    el = [str(e).strip().upper() for e in el]
    e1 = [e[:1] for e in el]; keep = [e != 'H' for e in e1]
    for a, b in bonds(crystal, e1): ax.plot(*zip(crystal[a], crystal[b]), color='#9ec3e6', lw=2, ls=(0, (4, 2)))
    for a, b in bonds(docked, e1):  ax.plot(*zip(docked[a], docked[b]), color='#666', lw=3)
    C, Dk = crystal[keep], docked[keep]; ek = [e for e, k in zip(el, keep) if k]
    ax.scatter(*C.T, c='#9ec3e6', s=45, alpha=.6, label='crystal')
    ax.scatter(*Dk.T, c=[EL.get(e, '#888') for e in ek], s=130, edgecolor='k', lw=.5, label='CIM pose')
    P = np.vstack([C, Dk]); c0 = P.mean(0); rad = np.abs(P - c0).max() + 1
    ax.set_xlim(c0[0]-rad, c0[0]+rad); ax.set_ylim(c0[1]-rad, c0[1]+rad); ax.set_zlim(c0[2]-rad, c0[2]+rad)
    ax.set_title(title, fontsize=11); ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
    ax.legend(fontsize=8, loc='upper left')

fig = plt.figure(figsize=(5 * len(MOLS), 5))
for i, p in enumerate(MOLS):
    d = B[p]; poses, r = decode(d); best = poses[int(r.argmin())]
    draw(fig.add_subplot(1, len(MOLS), i + 1, projection='3d'),
         d["lig_coords"], best, d["lig_elements"], f"{p}   mRMSD {r.min():.2f} Å")
plt.tight_layout()
os.makedirs(os.path.join(ROOT, "assets"), exist_ok=True)
fig.savefig(os.path.join(ROOT, "assets", "docking_demo.png"), dpi=110, bbox_inches="tight")
plt.show()

## 8. Interactive 3-D view

The crystal ligand (cyan) and the best CIM pose (orange) inside the binding pocket --
**drag to rotate, scroll to zoom.** The receptor is a transparent cartoon and the
pocket side-chains are thin sticks. Call `view3d("1LRH")` for the other target, or
`view3d("1N2J", pose=k)` to inspect any pooled pose.

In [ ]:
import py3Dmol

def _xyz(elements, coords):
    rows = [str(len(coords)), ""]
    for e, c in zip(elements, coords):
        rows.append(f"{str(e).strip().title():<2} {c[0]:.4f} {c[1]:.4f} {c[2]:.4f}")
    return "\n".join(rows)

def _receptor(pdb):
    return "".join(l[:66] + "\n" for l in open(f"{ROOT}/data/{pdb}_receptor.pdbqt")
                   if l.startswith(("ATOM", "HETATM")))

def view3d(pdb, pose=None, width=760, height=520):
    d = B[pdb]; poses, rmsd = decode(d)
    k = int(rmsd.argmin()) if pose is None else int(pose)
    v = py3Dmol.view(width=width, height=height)
    v.addModel(_receptor(pdb), "pdb")                              # model 0: protein
    v.addModel(_xyz(d["lig_elements"], d["lig_coords"]), "xyz")    # model 1: crystal pose
    v.addModel(_xyz(d["lig_elements"], poses[k]), "xyz")          # model 2: CIM pose
    v.setStyle({"model": 0}, {"cartoon": {"color": "spectrum", "opacity": 0.55}})
    v.addStyle({"model": 0, "within": {"distance": 4.5, "sel": {"model": 1}}},
               {"stick": {"colorscheme": "whiteCarbon", "radius": 0.1}})   # pocket side-chains
    v.setStyle({"model": 1}, {"stick": {"colorscheme": "cyanCarbon", "radius": 0.16}})
    v.setStyle({"model": 2}, {"stick": {"colorscheme": "orangeCarbon", "radius": 0.2},
                              "sphere": {"scale": 0.25}})
    v.zoomTo({"model": 1}); v.zoom(0.6); v.setBackgroundColor("white")
    v.show()

view3d("1N2J")   # try view3d("1LRH") or view3d("1N2J", pose=3)

## The classical solver

The identical QUBO also runs on Kaiwu's CPU simulated-annealing optimiser,
`kw.classical.SimulatedAnnealingOptimizer(...)` (wrapped in `backends.solve_sa`) --
useful for developing the encoding before spending CIM quota.

## References

- Wei *et al.*, "A versatile coherent Ising computing platform," *Light: Science &
  Applications* (2026) 15:74. DOI 10.1038/s41377-025-02178-1.
- Zha *et al.*, "Encoding Molecular Docking for Quantum Computers," *JCTC* (2024).
  DOI 10.1021/acs.jctc.3c00943.